In [24]:
from sentence_transformers import SentenceTransformer
import pandas as pds
from datasets import load_dataset
import os
from dotenv import dotenv_values
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import PointStruct, Document
from groq import Groq
import uuid
from google import genai

In [ ]:
# Load environment variables
config = dotenv_values(".env")
HF_TOKEN  = config.get("HF_TOKEN")
QDRANT_CLOUD_API_KEY = config.get("QDRANT_CLOUD_API_KEY")
QDRANT_CLOUD_ENDPOINT = config.get("QDRANT_CLOUD_ENDPOINT")
GROQ_API_KEY  = config["GROQ_API_KEY"]

# Make HuggingFace token available to the transformers library
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# Add .env Varaible Here If you are Using Google Colab



In [6]:
# Hugging Face Authentification
from huggingface_hub import login
login(token=HF_TOKEN)

In [7]:
collection_name = "Morrocan_Chat_Culture"
# Connection With QDrant Cloud
client_qdrant = QdrantClient(
    url=QDRANT_CLOUD_ENDPOINT,
    api_key=QDRANT_CLOUD_API_KEY,
    cloud_inference=True
)

In [20]:
# Collection Creation - Collection That Support Hybrid Search
client_qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(
                    distance=models.Distance.COSINE,
                    size=384,
        ),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

/tmp/ipykernel_2385/620268484.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client_qdrant.recreate_collection(


True

In [11]:
# client_qdrant.delete_collection(collection_name=collection_name)

In [8]:
# Connection Wth Groq
client_groq = Groq(api_key=GROQ_API_KEY)

In [18]:
# Load Dataset From Hugging Face
dataset = load_dataset(
    "atlasia/Atlaset",
    split="train",
    streaming=True # we do not download this dataset we create object refrence o dataset
).select_columns(["text", "word_count"])

In [ ]:
# Add Doocuments (group of tokens) to Cloud
def add_documents_to_qdrant(documents):
    client_qdrant.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=uuid.uuid4().hex,
                vector={
                    "dense": models.Document(
                        text=doc,
                        model="sentence-transformers/all-MiniLM-L6-v2",
                    ),
                    "sparse": models.Document(
                        text=doc,
                        model="Qdrant/bm25",
                    ),
                },
                payload={"text": doc},
            )
            for doc in documents
        ]
    )

In [11]:
# Dense Search
def dense_search(query: str) -> list[models.ScoredPoint]:
    response = client_qdrant.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="sentence-transformers/all-MiniLM-L6-v2",
        ),
        using="dense",
        limit=3,
    )
    for res in response.points:
        print(res.payload)
dense_search("تكلم شوية على مغرب")

{'text': 'كرة القدم في المغرب تستمر في النمو مع ظهور مواهب جديدة في الأندية.'}
{'text': 'يحب الكثير من المغاربة كرة القدم ويتابعون المباريات كل أسبوع.'}
{'text': 'لاعب كرة القدم الجيد يحتاج إلى التدريب والعمل الجماعي والانضباط.'}


In [12]:
# Sparse Search
def sparse_search(query: str) -> list[models.ScoredPoint]:
    response = client_qdrant.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model="Qdrant/bm25",
        ),
        using="sparse",
        limit=3,
    )
    for res in response.points:
        print(res.payload)
sparse_search("تكلم شوية على مغرب")

In [13]:
# Hybrid Search with Reciprocal Rank Fusion
'''
    Qdrant Combine Keyword Search with Semantic Search
    - Step1: Keyword Search to Get Relevent Chunks
    - Step2: Semantic Search to Get Also Relevent Chunks
    - Fusion of Two Relevent Chunks: By USING RRF = Reciprocal Rank Fusion
'''

def rrf_search(query: str) -> list[models.ScoredPoint]:
    response = client_qdrant.query_points(
        collection_name=collection_name,
        prefetch=[
            # keyWord Search - Sparse Vector
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="Qdrant/bm25",
                ),
                using="sparse",
                limit=3,
            ),
            # Semantic Search - Dense Vector
            models.Prefetch(
                query=models.Document(
                    text=query,
                    model="sentence-transformers/all-MiniLM-L6-v2",
                ),
                using="dense",
                limit=3,
            )
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=3,
    )
    tokens = []
    for res in response.points:
        tokens.append(res.payload["text"])
    return tokens
rrf_search("تكلم شوية على مغرب")

['كرة القدم في المغرب تستمر في النمو مع ظهور مواهب جديدة في الأندية.',
 'يحب الكثير من المغاربة كرة القدم ويتابعون المباريات كل أسبوع.',
 'لاعب كرة القدم الجيد يحتاج إلى التدريب والعمل الجماعي والانضباط.']

In [21]:
# PIPELINE:

BATCH_SIZE = 10
total_tokens = 100
counter = 0
start_id = 0
sentences = []
for row in dataset:
    # print(row)
    if total_tokens <= start_id:
        break
    if (row["word_count"] >= 50):
        sentences.append(row["text"])
        counter += 1
    else: continue
    if (BATCH_SIZE <= counter):
        # Push The sentences into QDrant VDB
        add_documents_to_qdrant(sentences)
        sentences = []
        counter = 0
        start_id += BATCH_SIZE

print(f"Total points upserted: {start_id}")

Total points upserted: 100


In [18]:
print("Hello World")

Hello World


In [ ]:
# Check Similarity Between Question And Chunks
def get_relevent_chunks(question):
    return rrf_search(question)

In [ ]:
# Get hypothetical Embedding Documents

def get_llm_documents(question):
    """Generate a short hypothetical documentation passage for `question`."""
    
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": (
                    "نتا مساعد كيعطي معلومات مفيدة. جاوب بالدارجة المغربية اللي ساهلة ومفهومة. "
                    "عطي جواب واضح ومختصر بلا إطالة." 
                ),
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0.7,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = [chunk.choices[0].delta.content for chunk in completion]
    res = [s for s in res if s]
    return "".join(res)

get_llm_documents("تكلم شوية على مغرب")


'المغرب بلاد حلوة ومليونية. عاصمة البلد مدينة الرباط، واللغة الرسمية هي العربية والفرنسية. \n\nالبلاد مساحة كبيرة واديات جميلة وتلال خضراء، والجبال الشاهقة. كما يوجد في المغرب البحر المتوسط والبحر الأبيض المتوسط.\n\nمغرب هو بلد غني بالثقافة والتراث، حيث يوجد العديد من المعالم التاريخية والسياحة. بعض الأمثلة هي مدينة مراكش والمدينة التاريخية بالرباط والدار البيضاء.\n\nمغرب هو بلد يعتبر من أغنى بلدان العالم بالثروة الشعبية والخيرات، حيث يوجد العديد من الأطباق التقليدية والمنتجات الحرفية والفنون التقليدية.'

In [48]:
# Get Reponse From LLM
def generate_response(context):
    pass

In [66]:
def get_hyde_embedding(hyde_documents, CHUNK_LEN=60):
    chunks = []
    chunks_temp = ""
    count = 0

    # Make Documents With Chunks
    list_words = hyde_documents.split(" ")

    # Make Chunks
    len_words = len(list_words)
    print(len_words)
    for i, word in enumerate(list_words):
        if (i >= len_words - 1):
            chunks.append(chunks_temp)
            break
        elif (count >= CHUNK_LEN):
            chunks.append(chunks_temp)
            count = 0
            chunks_temp = ""
        chunks_temp += word + " "
        count += 1
    
    return chunks


In [50]:
def search_by_cosin(relevent_chunks, hyde_documents_chunks):
    pass

In [ ]:
# Define Question
question = "نص مصاريف المخزن كانت كتمشي لبرا باش يخلصو الغرامات ديال الحرب و يشريو السلاح"

# Get hypothetical Embedding Documents
hyde_documents = get_llm_documents(question=question)

# Get Chunks from Hyde Documents
hyde_documents_chunks = get_hyde_embedding(hyde_documents)

# Get Relevent Chunks From Qdart
relevent_chunks = get_relevent_chunks(question=question)

# Merge Hypothetical Documents and Relevents Documents via RRF (Reciprocal Rank Fusion)



# Filtre To Get Context With Cosin Similarity
# filtred_documents = search_by_cosin(relevent_chunks, hyde_documents_chunks)
# context = filtred_documents

# # Get Response
# response = generate_response(context=context)
# print(response)



['اللي كتب مقالة عن تمويل الحرب بعنوان كيفية دفع ثمن الحرب', 'شحال من جولات كاينين فالدرافت ديال الـم.ل.ب.', 'اشنو النقود اللي كايستخدمو في كوالالمبور؟ ', 'شحال من ورقة ديال ألف دولار كاينة فالتداول؟ ', 'طوابع الرسوم المخصصة للحكومة المركزية طبعت في', 'اللي كايصنع النقود الورقية', 'اللي كايصنع النقود الورقية', 'المبلغ النقدي اللي كايقترح فيه المقيّم على العقار هو', 'شكون كتب حزم المشاكل ديالك في كيس العتاد ديالك', 'الللي كايموتو مع المال في الهروب من السجن']


In [ ]:
# Hybrid Search


In [ ]:
# Hyde Architect

In [ ]:
# Get Relevent Documents

In [ ]:
# Get Context and Give LLM Context and Get Response